In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
def validate_vacation_capacity(staff_vacations, start_date, total_days, min_required_staff=3):
    """
    Scans the calendar before generating the schedule to ensure 
    there are enough workers available every single day.
    """
    print("Checking vacation lists for staffing bottlenecks...")
    bottlenecks_found = False
    
    for day_idx in range(total_days):
        current_date = start_date + timedelta(days=day_idx)
        date_str = current_date.strftime("%Y-%m-%d")
        
        # Find out who is on vacation on this specific day
        on_vacation = [worker for worker, dates in staff_vacations.items() if date_str in dates]
        available_count = len(staff_vacations) - len(on_vacation)
        
        # If available staff drops below 3, we have a bottleneck
        if available_count < min_required_staff:
            print(f"⚠️  CRITICAL BOTTLENECK on {date_str} ({current_date.strftime('%A')}):")
            print(f"    Only {available_count} workers available! Minimum required is {min_required_staff}.")
            print(f"    Workers on vacation: {', '.join(on_vacation)}\n")
            bottlenecks_found = True
            
    if bottlenecks_found:
        print("❌ Schedule generation halted. Please adjust conflicting vacation requests above.")
        return False
    
    print("✅ No bottlenecks found! All dates have sufficient staffing levels.\n")
    return True


def generate_hospital_schedule_secure(start_date_str, total_days=180):
    # 1. Define staff and vacation schedules
    # TEST CASE: Dr. Alice, Nurse Bob, and Dr. Charlie all requested Dec 25 off.
    # With 8 staff total, 3 off leaves 5 available, which passes (5 >= 3).
    # If 6 people request it off, it will trigger the bottleneck error.
    staff_vacations = {
        "Dr. Alice Smith": {"2026-09-15", "2026-09-16", "2026-12-25"},
        "Nurse Bob Jones": {"2026-12-25"},
        "Dr. Charlie Brown": {"2026-10-01", "2026-10-02", "2026-12-25"},
        "Nurse Diana Prince": set(),
        "Dr. Evan Wright": {"2026-12-24", "2026-12-25", "2026-12-25", "2026-12-26"},
        "Nurse Fiona Gallagher": set(),
        "Dr. George Clark": {"2026-11-26"},
        "Nurse Hannah Abbott": {"2026-11-26", "2026-12-25"} 
    }
    
    workers = list(staff_vacations.keys())
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    
    # 2. Run the validation check first
    if not validate_vacation_capacity(staff_vacations, start_date, total_days):
        return  # Exit early to prevent scheduling errors or code crashes
        
    # 3. Schedule generation code proceeds safely...
    shift_counts = {w: {'Day_Shift': 0, 'Night_Shift': 0, 'Weekend_Shifts': 0, 'Total': 0} for w in workers}
    last_night_worker = None
    schedule_data = []

    for day_idx in range(total_days):
        current_date = start_date + timedelta(days=day_idx)
        date_str = current_date.strftime("%Y-%m-%d")
        day_name = current_date.strftime("%A")
        is_weekend = day_name in ['Saturday', 'Sunday']
        
        active_workers = [w for w in workers if date_str not in staff_vacations[w]]
        
        # --- Day Shift ---
        available_for_day = [w for w in active_workers if w != last_night_worker]
        if is_weekend:
            available_for_day.sort(key=lambda w: (shift_counts[w]['Weekend_Shifts'], shift_counts[w]['Day_Shift'], shift_counts[w]['Total']))
        else:
            available_for_day.sort(key=lambda w: (shift_counts[w]['Day_Shift'], shift_counts[w]['Total']))
            
        assigned_day_worker = available_for_day[0]
        shift_counts[assigned_day_worker]['Day_Shift'] += 1
        shift_counts[assigned_day_worker]['Total'] += 1
        if is_weekend: shift_counts[assigned_day_worker]['Weekend_Shifts'] += 1
        
        # --- Night Shift ---
        available_for_night = [w for w in active_workers if w != assigned_day_worker and w != last_night_worker]
        if is_weekend:
            available_for_night.sort(key=lambda w: (shift_counts[w]['Weekend_Shifts'], shift_counts[w]['Night_Shift'], shift_counts[w]['Total']))
        else:
            available_for_night.sort(key=lambda w: (shift_counts[w]['Night_Shift'], shift_counts[w]['Total']))
            
        assigned_night_worker = available_for_night[0]
        shift_counts[assigned_night_worker]['Night_Shift'] += 1
        shift_counts[assigned_night_worker]['Total'] += 1
        if is_weekend: shift_counts[assigned_night_worker]['Weekend_Shifts'] += 1
        
        schedule_data.append({
            'Date': date_str, 'Day of Week': day_name,
            'Day Shift (7 AM - 7 PM)': assigned_day_worker,
            'Night Shift (7 PM - 7 AM)': assigned_night_worker
        })
        last_night_worker = assigned_night_worker

        # 3. Export to CSV file
        csv_filename = "hospital_schedule_with_rest.csv"
        fields = ['Date', 'Day of Week', 'Day Shift (7 AM - 7 PM)', 'Night Shift (7 PM - 7 AM)']
    
        with open(csv_filename, mode='w', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()
            writer.writerows(schedule_data)
 
    print("Generation complete! Schedule saved to hospital_schedule_with_rest.csv")
    
    return pd.DataFrame(schedule_data)

In [ ]:
# Run the generator starting tomorrow
#    return pd.DataFrame(schedule_data)
dat = generate_hospital_schedule_secure("2026-09-13")

In [ ]:
dat.columns = ['date','day_of_week','day_shift','night_shift']
dat.iloc[:20]
#pd.pivot_table(dat,index = 'day_of_week',columns = 'night_shift',aggfunc = 'size')

In [ ]:
dat[((dat.day_of_week == 'Thursday') ) & (dat.night_shift == 'Dr. Alice Smith')]
